In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
!pip install -q jupyter_bbox_widget roboflow dataclasses-json supervision

import os
from google.colab import drive
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
import cv2
import supervision as sv
import numpy as np

drive.mount('/content/drive')
# !nvidia-smi

HOME = os.getcwd()
# print("HOME:", HOME)



CHECKPOINT_PATH = os.path.join(HOME, "drive", "MyDrive", "SAM_models", "sam_vit_l_0b3195.pth")
IMAGE_PATH = os.path.join(HOME, "drive", "MyDrive","512x512NbTi")
# print(CHECKPOINT_PATH, "; exist:", os.path.isfile(CHECKPOINT_PATH))
# print(IMAGE_PATH, "; exist:", os.path.isfile(IMAGE_PATH))

# !mkdir -p {HOME}/data
#make sure image pixle size is 512X512: Use https://www.freeconvert.com/tiff-converter
# Go into the Files tab on the left and insert your images you want to analyze into the data folder

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_TYPE = "vit_l"
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
mask_generator = SamAutomaticMaskGenerator(sam)

# print("Image Path: " + IMAGE_PATH)
for filename in os.listdir(IMAGE_PATH):
  # print("filename: " + filename)
  current_file = os.path.join(IMAGE_PATH,filename)
  image_bgr = cv2.imread(current_file)
  image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
  sam_result = mask_generator.generate(image_rgb)

  # print(sam_result[0].keys())

  mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)

  detections = sv.Detections.from_sam(sam_result=sam_result)

  annotated_image = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)

  # cv2.imwrite(current_file, annotated_image)
  cv2.imwrite(os.path.join(HOME, "drive", "MyDrive","NbTi_seg",filename), annotated_image)
  # sv.plot_images_grid(
  #     images=[image_bgr, annotated_image],
  #     grid_size=(1, 2),
  #     titles=['Original '+ filename + 'image','Segmented '+ + 'image']
  # )

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.3/158.3 kB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.7 MB/s eta 0:00:00
Mounted at /content/drive
